# 📊 Notebook 08: Visualisierung

**Szenario**: Erstelle einen Report für deinen Vorgesetzten – alle Erkenntnisse aus der EDA als saubere Charts.

**Lernziele:**
- matplotlib: Figure, Axes, Subplots
- seaborn: Statistik-Plots (pairplot, heatmap, distplot)
- plotly: Interaktive Charts
- Wann welchen Plot?

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Daten laden
df_churn = pd.read_csv(BASE_URL + "customer_churn.csv")
df_churn['total_charges'] = pd.to_numeric(df_churn['total_charges'], errors='coerce')
df_movies = pd.read_csv(BASE_URL + "movies.csv")
df_flights = pd.read_csv(BASE_URL + "flights.csv")

print('Datensätze geladen!')

## 1️⃣ matplotlib – Die Basis

In [ ]:
# Multi-Panel Dashboard mit GridSpec
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: Churn-Verteilung
ax1 = fig.add_subplot(gs[0, 0])
churn_counts = df_churn['churn'].value_counts()
colors_churn = ['#0389CF', '#E8792F']
ax1.bar(['Kein Churn', 'Churn'], churn_counts.values, color=colors_churn, edgecolor='white', linewidth=1.5)
ax1.set_title('Churn-Verteilung', fontweight='bold')
ax1.set_ylabel('Anzahl Kunden')
for i, (bar, val) in enumerate(zip(ax1.patches, churn_counts.values)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center', fontweight='bold')

# Panel 2: Monthly Charge nach Contract Type
ax2 = fig.add_subplot(gs[0, 1])
df_churn.boxplot(column='monthly_charge', by='contract_type', ax=ax2)
ax2.set_title('Monatl. Kosten je Vertragstyp')
plt.sca(ax2); plt.title('Monatl. Kosten je Vertragstyp', fontweight='bold')
ax2.set_xlabel('')
plt.gcf().suptitle('')

# Panel 3: Histogramm Tenure
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(df_churn['tenure_months'], bins=30, color='#0389CF', alpha=0.7, edgecolor='white')
ax3.set_title('Kundenlaufzeit', fontweight='bold')
ax3.set_xlabel('Monate')

# Panel 4: Genres (Movies)
ax4 = fig.add_subplot(gs[1, 0])
genre_avg = df_movies.groupby('genre')['rating'].mean().sort_values(ascending=True)
ax4.barh(genre_avg.index, genre_avg.values, color='#0389CF')
ax4.set_title('Ø Rating je Genre', fontweight='bold')
ax4.set_xlim(0, 10)

# Panel 5: Scatter Budget vs Revenue
ax5 = fig.add_subplot(gs[1, 1])
scatter = ax5.scatter(df_movies['budget_mio'], df_movies['revenue_mio'],
                      alpha=0.4, c=df_movies['rating'], cmap='Blues', s=20)
ax5.set_title('Budget vs. Revenue', fontweight='bold')
ax5.set_xlabel('Budget (Mio $)')
ax5.set_ylabel('Revenue (Mio $)')
plt.colorbar(scatter, ax=ax5, label='Rating')

# Panel 6: Flugverspätungen
ax6 = fig.add_subplot(gs[1, 2])
delay_clean = df_flights['dep_delay'].dropna()
delay_clean[delay_clean.between(-30, 120)].hist(bins=40, ax=ax6, color='#E8792F', alpha=0.7, edgecolor='white')
ax6.set_title('Departure Delay', fontweight='bold')
ax6.set_xlabel('Minuten')

plt.savefig("/tmp/dashboard_matplotlib.png", dpi=100, bbox_inches='tight')
plt.show()
print('✅ Dashboard gespeichert!')

## 2️⃣ seaborn – Statistische Visualisierungen

In [ ]:
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap: Korrelationsmatrix
numeric = df_churn.select_dtypes('number').dropna()
corr = numeric.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[0], square=True, cbar_kws={'shrink':0.8})
axes[0].set_title('Korrelationsmatrix – Customer Churn', fontweight='bold')

# Violin + Swarm: Monthly Charge nach Churn
sns.violinplot(data=df_churn, x='churn', y='monthly_charge',
               palette=['#0389CF','#E8792F'], ax=axes[1], inner='quartile')
axes[1].set_title('Monthly Charge nach Churn', fontweight='bold')
axes[1].set_xticklabels(['Kein Churn (0)', 'Churn (1)'])

plt.tight_layout()
plt.show()

## 3️⃣ Plotly – Interaktive Charts

In [ ]:
# Interaktiver Scatter: Budget vs Revenue
fig = px.scatter(df_movies,
    x='budget_mio', y='revenue_mio',
    color='genre', size='votes',
    hover_data=['title','year','rating'],
    title='📊 Budget vs. Revenue nach Genre (interaktiv)',
    labels={'budget_mio':'Budget (Mio $)', 'revenue_mio':'Revenue (Mio $)'},
    color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Interaktives Balkendiagramm: Churn nach Vertragstyp
churn_by_contract = df_churn.groupby('contract_type')['churn'].mean().reset_index()
churn_by_contract.columns = ['contract_type', 'churn_rate']
churn_by_contract['churn_rate_pct'] = (churn_by_contract['churn_rate'] * 100).round(1)

fig = px.bar(churn_by_contract,
    x='contract_type', y='churn_rate_pct',
    title='📉 Churn-Rate nach Vertragstyp',
    color='churn_rate_pct',
    color_continuous_scale='Oranges',
    labels={'churn_rate_pct':'Churn-Rate (%)','contract_type':'Vertragstyp'},
    text='churn_rate_pct')
fig.update_traces(texttemplate='%{text}%')
fig.update_layout(height=400)
fig.show()

## 💡 Wann welchen Plot?

| Frage | Plot-Typ |
|-------|----------|
| Wie ist eine Variable verteilt? | Histogramm, KDE, Violinplot |
| Gibt es Ausreißer? | Boxplot |
| Zusammenhang zweier Variablen? | Scatter-Plot |
| Kategorien vergleichen? | Bar-Chart, Grouped Bar |
| Entwicklung über Zeit? | Line-Chart |
| Korrelationen zwischen vielen Variablen? | Heatmap |
| Anteil am Ganzen? | Pie-Chart (sparsam einsetzen!) |

## ✅ Challenges

1. Erstelle ein Seaborn Pairplot für die numerischen Spalten von customer_churn, gefärbt nach `churn`
2. Erstelle ein interaktives Plotly Line-Chart: Durchschnittliche Filmrating nach Jahr
3. Erstelle ein Balkendiagramm: Top 5 Airlines nach durchschnittlicher Verspätung

In [ ]:
# Deine Lösung:


---
**Weiter mit:** `09_Fallstudie_Customer_Churn.ipynb`